# Week 12 Lab — Causal Informatics: Interventions and Decision Impact
**ISE 16:540 · AI-Enabled Informatics · Rutgers University · Spring 2026**

---

**How to run:** Click **Runtime → Run all**. Work through exercises in order.

| Exercise | What you do |
|---|---|
| 1 — Confounding Simulation | Generate confounded data and see how naive correlation misleads |
| 2 — DoWhy Causal Analysis | Run the four-step DoWhy workflow on the same data |
| 3 — Assignment 5 Deliverable | Apply to your own scenario and write the decision recommendation |


In [ ]:
# Install DoWhy (required this week — it is the course tool for causal inference)
!pip install dowhy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
print("Ready.")

---
# Exercise 1 — Confounding Simulation

We generate a dataset where **product complexity** (the confounder) causes both
**supplier lead time** (the treatment) and **defect rate** (the outcome).

The raw correlation between lead time and defects will look strong.
After adjusting for complexity, the true causal effect will be much smaller.


In [ ]:
n = 2000

# Confounder: product complexity (0=simple, 1=complex)
complexity = np.random.binomial(1, 0.45, n)

# Treatment: supplier lead time (days) — caused partly by complexity
lead_time = 5 + 3 * complexity + np.random.normal(0, 1.5, n)

# Outcome: defect rate (%) — caused by complexity AND weakly by lead time
# True causal effect of lead_time on defects: 0.05 per day
# Complexity also drives defects strongly (0.8 percentage points)
defect_rate = 1.0 + 0.05 * lead_time + 0.8 * complexity + np.random.normal(0, 0.3, n)

df = pd.DataFrame({
    "lead_time":   lead_time,
    "defect_rate": defect_rate,
    "complexity":  complexity,
})

print(f"Dataset: {len(df)} rows")
print(df.describe().round(2))

In [ ]:
# --- Naive observational correlation ---
from numpy.polynomial.polynomial import polyfit

naive_corr = df["lead_time"].corr(df["defect_rate"])
print(f"Naive correlation (lead_time vs defect_rate): r = {naive_corr:.3f}")

# Naive regression: defect ~ lead_time only (ignoring complexity)
X_naive = np.column_stack([np.ones(n), df["lead_time"]])
beta_naive = np.linalg.lstsq(X_naive, df["defect_rate"], rcond=None)[0]
print(f"Naive OLS coefficient on lead_time: {beta_naive[1]:.4f} (true = 0.05)")
print(f"Naive OLS intercept:                {beta_naive[0]:.4f}")
print()
print("The naive model OVERESTIMATES the effect of lead time because")
print("it absorbs the confounding influence of product complexity.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor("#0f1117")

# Panel 1: raw scatter
ax = axes[0]
ax.set_facecolor("#1a1d27")
colors = ["#6366f1" if c == 0 else "#f59e0b" for c in df["complexity"]]
ax.scatter(df["lead_time"], df["defect_rate"], c=colors, alpha=0.3, s=12)
x_line = np.linspace(df["lead_time"].min(), df["lead_time"].max(), 100)
ax.plot(x_line, beta_naive[0] + beta_naive[1]*x_line, color="#ef4444", lw=2, label="Naive fit")
ax.set_title("Observational: Lead Time vs Defect Rate", color="#e2e8f0", fontsize=11)
ax.set_xlabel("Lead Time (days)", color="#94a3b8")
ax.set_ylabel("Defect Rate (%)", color="#94a3b8")
ax.tick_params(colors="#94a3b8")
for spine in ax.spines.values(): spine.set_edgecolor("#2e3352")
ax.legend(loc="upper left", fontsize=9)

# Panel 2: by complexity group
ax2 = axes[1]
ax2.set_facecolor("#1a1d27")
for cval, clabel, col in [(0, "Simple", "#6366f1"), (1, "Complex", "#f59e0b")]:
    mask = df["complexity"] == cval
    ax2.scatter(df.loc[mask,"lead_time"], df.loc[mask,"defect_rate"],
               c=col, alpha=0.35, s=12, label=clabel)
    X_grp = np.column_stack([np.ones(mask.sum()), df.loc[mask,"lead_time"]])
    b = np.linalg.lstsq(X_grp, df.loc[mask,"defect_rate"], rcond=None)[0]
    x_g = np.linspace(df.loc[mask,"lead_time"].min(), df.loc[mask,"lead_time"].max(), 50)
    ax2.plot(x_g, b[0]+b[1]*x_g, color=col, lw=2)
ax2.set_title("Within-group (adjusted for complexity)", color="#e2e8f0", fontsize=11)
ax2.set_xlabel("Lead Time (days)", color="#94a3b8")
ax2.set_ylabel("Defect Rate (%)", color="#94a3b8")
ax2.tick_params(colors="#94a3b8")
for spine in ax2.spines.values(): spine.set_edgecolor("#2e3352")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("confounding_demo.png", dpi=120, facecolor="#0f1117", bbox_inches="tight")
plt.show()
print("Left: naive model overestimates the slope.")
print("Right: within each complexity group, the true (smaller) slope appears.")

### YOUR WORK — Exercise 1 Questions

**Q1. The naive correlation is r ≈ 0.65 and the naive OLS coefficient is much larger than 0.05 (the true causal effect). Explain in one sentence why the naive estimate is inflated.**

*Answer:*

---

**Q2. In the within-group plot (right panel), the slope within each complexity group is closer to 0.05. What does this tell you about the role of complexity?**

*Answer:*

---

**Q3. A procurement manager uses the naive model and concludes: "every extra day of lead time adds 0.3% to defect rate — switch all orders to fast suppliers." What is wrong with this conclusion?**

*Answer:*

---
# Exercise 2 — DoWhy Causal Analysis

Run the four-step DoWhy workflow on the same data:
1. **Model** — define the causal DAG
2. **Identify** — find a valid identification strategy
3. **Estimate** — compute the causal effect
4. **Refute** — test robustness of the estimate


In [ ]:
import dowhy
from dowhy import CausalModel

# Step 1: Define the causal model
# Edges encode our domain knowledge: what causes what
model = CausalModel(
    data=df,
    treatment="lead_time",
    outcome="defect_rate",
    # Specify the causal graph as GML or edge list
    common_causes=["complexity"],  # complexity is the confounder
)

print("Causal model defined.")
print(f"Treatment: {model.treatment_name}")
print(f"Outcome:   {model.outcome_name}")
print(f"Common causes (confounders): {model.common_causes}")

In [ ]:
# Step 2: Identify — find a valid estimation strategy
# DoWhy searches the DAG for how to express P(Y|do(T)) using observables
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

print("Identified estimand:")
print(identified_estimand)

In [ ]:
# Step 3: Estimate the causal effect
# Using linear regression adjustment (back-door criterion)
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    test_significance=True,
)

print("=== Causal Effect Estimate ===")
print(estimate)
print(f"\nATE (Average Treatment Effect): {estimate.value:.4f}")
print(f"True causal effect:              0.0500")
print(f"Naive OLS estimate:              ~0.30 (inflated by confounding)")
print()
print("DoWhy recovers the true causal effect by adjusting for the confounder.")

In [ ]:
# Step 4a: Refutation — add a random common cause
# If our causal model is correct, adding a random variable should NOT change the estimate
refute_random = model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="random_common_cause",
)
print("Refutation test 1 — Random Common Cause:")
print(refute_random)
print()
print("If the estimate is robust, the new estimate should be close to the original.")

In [ ]:
# Step 4b: Refutation — placebo treatment
# Replace the real treatment with a random variable
# The causal effect should go to zero
refute_placebo = model.refute_estimate(
    identified_estimand,
    estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
)
print("Refutation test 2 — Placebo Treatment:")
print(refute_placebo)
print()
print("With a random placebo treatment, the effect should be near zero.")
print("If it is not, your model may be picking up spurious associations.")

### YOUR WORK — Exercise 2 Questions

**Q1. The DoWhy ATE should be close to 0.05 (the true causal effect). The naive OLS estimate was much larger. Explain in one sentence what adjustment step produced the correction.**

*Answer:*

---

**Q2. The placebo treatment refutation should return an effect near zero. Why is this a meaningful check? What would it mean if the placebo estimate were NOT near zero?**

*Answer:*

---

**Q3. Based on the causal estimate (ATE ≈ 0.05), what should a procurement manager actually decide? Compare to the naive recommendation.**

*Answer:*

---
# Exercise 3 — Your Own Causal Mini-Study
## This is your Assignment 5 deliverable

Apply the DoWhy workflow to your own scenario.
You may use the provided starter dataset (manufacturing process data) OR bring your own data.

**Your scenario must have:**
- A clearly defined treatment variable (something you could plausibly intervene on)
- A clearly defined outcome variable (what you care about changing)
- At least one confounder (a variable that causes both treatment and outcome)


In [ ]:
# Option A: Use the starter dataset (manufacturing process)
# Treatment:  machine_setting (0=standard, 1=high-precision)
# Outcome:    quality_score (0-100)
# Confounder: operator_shift (0=day, 1=night) — night shift uses different raw material batches

n_s = 1500
np.random.seed(99)

operator_shift = np.random.binomial(1, 0.40, n_s)  # 40% night shift

# Machine setting: night shift operators more likely to use high-precision (more careful)
p_high = 0.3 + 0.3 * operator_shift
machine_setting = np.random.binomial(1, p_high, n_s)

# Quality score:
#  - True causal effect of high-precision setting: +4 points
#  - Night shift has lower raw material quality on average: -6 points
quality_score = (
    75
    + 4.0 * machine_setting        # true causal effect
    - 6.0 * operator_shift         # confounder effect
    + np.random.normal(0, 3, n_s)
)

df_mfg = pd.DataFrame({
    "machine_setting": machine_setting,
    "quality_score":   quality_score,
    "operator_shift":  operator_shift,
})

print(f"Manufacturing dataset: {len(df_mfg)} rows")
print(df_mfg.describe().round(2))
print()
# Quick check: naive vs. adjusted
naive_diff = df_mfg.groupby("machine_setting")["quality_score"].mean().diff().iloc[1]
print(f"Naive difference (high vs. standard setting): {naive_diff:.2f} points")
print(f"True causal effect:                          +4.00 points")
print()
print("Notice how the confounder (shift) suppresses the apparent effect.")
print("Night shift operators use high-precision more often but get worse scores")
print("due to raw material quality — masking the real benefit of the setting.")

In [ ]:
# YOUR WORK — Step 1: Define your causal model
# If using the starter dataset, fill in the treatment, outcome, and common_causes below.
# If using your own dataset, replace df_mfg with your DataFrame.

YOUR_DATA       = df_mfg            # ← replace with your DataFrame if using own data
YOUR_TREATMENT  = "machine_setting" # ← treatment variable name
YOUR_OUTCOME    = "quality_score"   # ← outcome variable name
YOUR_CONFOUNDERS = ["operator_shift"] # ← list of confounder variable names

my_model = CausalModel(
    data=YOUR_DATA,
    treatment=YOUR_TREATMENT,
    outcome=YOUR_OUTCOME,
    common_causes=YOUR_CONFOUNDERS,
)
print("Your causal model defined.")
print(f"  Treatment:  {my_model.treatment_name}")
print(f"  Outcome:    {my_model.outcome_name}")
print(f"  Confounders: {my_model.common_causes}")

In [ ]:
# YOUR WORK — Steps 2-4: Identify, Estimate, Refute

# Step 2: Identify
my_estimand = my_model.identify_effect(proceed_when_unidentifiable=True)
print("Identification complete.")

# Step 3: Estimate
my_estimate = my_model.estimate_effect(
    my_estimand,
    method_name="backdoor.linear_regression",
    test_significance=True,
)
print(f"\nATE estimate: {my_estimate.value:.4f}")
print(my_estimate)

# Step 4: Refute (run at least one test)
my_refutation = my_model.refute_estimate(
    my_estimand,
    my_estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
)
print("\nPlacebo refutation:")
print(my_refutation)

In [ ]:
# Verification — all three checks must pass before submitting
import math

checks = [
    ("Causal model defined", my_model is not None),
    ("Estimand found",        my_estimand is not None),
    ("ATE is a finite number",not math.isnan(my_estimate.value) and not math.isinf(my_estimate.value)),
    ("Refutation run",        my_refutation is not None),
]
all_ok = True
for label, ok in checks:
    print(f"  {chr(9989) if ok else chr(10060)} {label}")
    if not ok: all_ok = False
print()
print("Ready to submit." if all_ok else "Fix failing checks before submitting.")

### YOUR WORK — Draw Your DAG

Describe your causal DAG here as a structured list. For each edge, explain the domain justification.

```
Example:
  operator_shift  →  machine_setting   (night shift more likely to use high-precision)
  operator_shift  →  quality_score     (night shift gets lower-quality raw materials)
  machine_setting →  quality_score     (high-precision directly improves surface finish)
```

**Your DAG edges:**

```
[YOUR EDGES HERE]
```

**Your treatment, outcome, confounder(s):**
- Treatment:
- Outcome:
- Confounder(s):
- Mediator(s) (if any):


### YOUR WORK — Decision Recommendation (~150 words)

Based on your causal estimate, write a decision recommendation.
Address all three points:
1. What action do you recommend, and what is the causal justification?
2. What would have gone wrong if you had used the raw observational correlation?
3. What is one assumption in your causal model that you are not certain is correct?

*Your recommendation:*

---
## Assignment 5 Submission Checklist

- [ ] **Runtime → Restart and run all** — every cell runs without errors
- [ ] Exercise 1: all three questions answered
- [ ] Exercise 2: all three questions answered; DoWhy ATE close to 0.05
- [ ] Exercise 3: verification cell shows all four checks passing
- [ ] Exercise 3: DAG edges specified with domain justification
- [ ] Exercise 3: 150-word decision recommendation written
- [ ] Submitted to Canvas → Assignments → Assignment 5

---
*ISE 16:540 · AI-Enabled Informatics · Week 12 Lab · Spring 2026 · Rutgers University*